# 1. 라이브러리 로드 및 모델 초기화

In [ ]:
import pandas as pd
import numpy as np
import torch
import urllib.request
import json
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer, TextClassificationPipeline
from torch.utils.data import Dataset
from konlpy.tag import Hannanum

# 토크나이저 및 사전학습 모델 로드 (다중작업 감성분석 모델)
MODEL_NAME = 'jhgan/ko-sroberta-multitask'
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertForSequenceClassification.from_pretrained(MODEL_NAME)

# 2. 딥러닝 감성 분류 모델 파인튜닝 (학습)

In [ ]:
# 데이터 로드 (기존 라벨링된 훈련용 데이터)
data = pd.read_csv('안유진.txt', sep='\t').dropna()

# sklearn을 활용한 깔끔한 Train/Test 분할 (8:2)
train_set, test_set = train_test_split(data, test_size=0.2, random_state=42)

# 데이터 추출 및 토크나이징
train_docs = train_set['document'].tolist()[:15000]
train_labels = train_set['label'].to_numpy()[:15000]
train_encodings = tokenizer(train_docs, return_tensors='pt', truncation=True, padding=True)

test_docs = test_set['document'].tolist()[:5000]
test_labels = test_set['label'].to_numpy()[:5000]
test_encodings = tokenizer(test_docs, return_tensors='pt', truncation=True, padding=True)

# PyTorch Dataset 클래스 정의
class PrepDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encoding = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encoding.items()}
        item['labels'] = torch.tensor(self.labels[idx]).long()
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = PrepDataset(train_encodings, train_labels)
test_dataset = PrepDataset(test_encodings, test_labels)

# Trainer 세팅 및 학습
training_args = TrainingArguments(
    output_dir='./checkpoint',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.0001,
    do_eval=True,
    evaluation_strategy='steps',
    fp16=True,
    learning_rate=2e-5,
    eval_steps=100,
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

# 3. 파이프라인 구축 및 평가

In [ ]:
# 파이프라인 생성
text_classifier = TextClassificationPipeline(
    tokenizer=tokenizer,
    model=model,
    device='cuda:0' if torch.cuda.is_available() else -1
)

# 모델 성능 평가 (Classification Report)
result = text_classifier(test_docs)
pred_labels = [int(res['label'].split('_')[-1]) for res in result]
print(classification_report(y_true=test_labels, y_pred=pred_labels))


# 4. 아티스트 시너지 분석 파이프라인 (크롤링 + 분류 + 형태소 분석)

In [ ]:
ID = "네이버_API_ID"
KEY = "네이버_API_KEY"

def analyze_artist_synergy(target_name):
    print(f"\n[{target_name}] 데이터 수집 및 분석 시작...")
    file_name = f"{target_name}.txt"

    # 1) 네이버 API 크롤링
    encText = urllib.parse.quote(target_name)
    with open(file_name, 'w', encoding='utf8') as f:
        for i in range(1, 1001, 100):
            url = f"https://openapi.naver.com/v1/search/news?query={encText}&display=100&start={i}"
            request = urllib.request.Request(url)
            request.add_header("X-Naver-Client-Id", ID)
            request.add_header("X-Naver-Client-Secret", KEY)

            try:
                response = urllib.request.urlopen(request)
                if response.getcode() == 200:
                    items = json.loads(response.read().decode('utf-8'))['items']
                    for item in items:
                        title = item['title'].replace('<b>', '').replace('</b>', '').replace('&quot;', '').replace('&apos;', '')
                        desc = item['description'].replace('<b>', '').replace('</b>', '').replace('&quot;', '').replace('&apos;', '')
                        f.write(title + '\n' + desc + '\n')
            except Exception as e:
                pass

    # 2) 데이터 로드 및 긍/부정 필터링
    with open(file_name, 'r', encoding='utf-8') as f:
        data_list = f.readlines()[:10000] # 상위 1만 줄만 사용

    label_1_sentences = []
    for sentence in data_list:
        if len(sentence.strip()) > 5: # 빈 줄 방지
            result = text_classifier(sentence[:512]) # 토큰 길이 제한
            if result[0]['label'] == 'LABEL_1':
                label_1_sentences.append(sentence)

    # 3) 형태소 분석 (명사 추출) 및 빈도 계산
    hannanum = Hannanum()
    words = []
    for sentence in label_1_sentences:
        tagged_words = hannanum.pos(sentence)
        for word, tag in tagged_words:
            if tag.startswith('N') and len(word) > 1: # 1글자 명사 제외
                words.append(word)

    # 4) 결과 출력 (Top 20)
    word_counts = Counter(words)
    sorted_words = word_counts.most_common(20)

    print(f"--- [{target_name}] 긍정 텍스트 내 동시출현 키워드 Top 20 ---")
    for word, count in sorted_words:
        print(f"{word}: {count}")

# 5. 최종 시너지 도출 실행

In [ ]:
analyze_artist_synergy('장원영')
analyze_artist_synergy('카리나')
analyze_artist_synergy('김채원')